* A 2000–2026 panel of 217 sovereign nations with annual cross-country rankings (1 = best, 217 = worst) across 11 international dimensions, integrating World Bank indicators and benchmark indices from UNDP, Transparency International, EIU, RSF, IEP, and Yale EPI.

* Harmonized for time-series and cross-sectional analysis, the dataset supports trend analysis, panel regression, clustering, and visualizations to evaluate development, governance, press freedom, peace, environment, and inform research and policy decisions.

* this is data analytics project with proper python analysis 


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("global_country_rankings_2000_2026.csv")
display(df.head())

,Country,Year,Region,Economic_Tier,Happiness_Rank,Global_Hunger_Rank,Human_Development_Rank,GDP_Per_Capita_Rank,Life_Expectancy_Rank,Corruption_Perception_Rank,Democracy_Rank,Gini_Rank,Press_Freedom_Rank,Global_Peace_Rank,Environmental_Performance_Rank
0,Afghanistan,2000,Asia,4,195,208,187,212,186,185,215,183,214,216,212
1,Afghanistan,2001,Asia,4,198,207,188,215,186,185,215,180,214,217,214
2,Afghanistan,2002,Asia,4,196,208,192,213,185,185,215,174,214,217,213
3,Afghanistan,2003,Asia,4,196,207,195,214,183,185,215,173,214,217,214
4,Afghanistan,2004,Asia,4,194,210,197,214,183,185,216,172,214,217,212


In [3]:
df.columns = (df.columns.str.strip().str.lower().str.replace(" ", "_"))
df.columns

Index(['country', 'year', 'region', 'economic_tier', 'happiness_rank',
       'global_hunger_rank', 'human_development_rank', 'gdp_per_capita_rank',
       'life_expectancy_rank', 'corruption_perception_rank', 'democracy_rank',
       'gini_rank', 'press_freedom_rank', 'global_peace_rank',
       'environmental_performance_rank'],
      dtype='object')

In [4]:
df.info()
print("--"* 20)
print(f"null chheck : \n{df.isnull().sum()}")
print("--"* 20)
print(f"duplicate check : \n{df.duplicated().sum()}")
print("--"* 20)
print(f" data shape : {df.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5859 entries, 0 to 5858
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   country                         5859 non-null   object
 1   year                            5859 non-null   int64 
 2   region                          5859 non-null   object
 3   economic_tier                   5859 non-null   int64 
 4   happiness_rank                  5859 non-null   int64 
 5   global_hunger_rank              5859 non-null   int64 
 6   human_development_rank          5859 non-null   int64 
 7   gdp_per_capita_rank             5859 non-null   int64 
 8   life_expectancy_rank            5859 non-null   int64 
 9   corruption_perception_rank      5859 non-null   int64 
 10  democracy_rank                  5859 non-null   int64 
 11  gini_rank                       5859 non-null   int64 
 12  press_freedom_rank              5859 non-null   

In [5]:
def economic_tier(val):
    if val == 1:
        return "high income"
    elif val ==2 :
        return "upper middle"
    elif val == 3 :
        return "lower middle"
    else:
        return "low"
df['economic_tier'] = df['economic_tier'].apply(economic_tier)

In [6]:
df['economic_tier'].value_counts()

economic_tier
high income     2322
upper middle    1593
lower middle    1269
low              675
Name: count, dtype: int64

In [7]:
# remove white space from object 
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

In [8]:
for col in df.select_dtypes(include='object').columns:
    if df[col].str.startswith(' ', na=False).any() or df[col].str.endswith(' ', na=False).any():
        print(f"Whitespace still exists in: {col}")

In [9]:
# check unique value in object column
for col in df.select_dtypes(include='object').columns:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(f"Unique Values: {df[col].nunique()}")
    print(df[col].value_counts())


Column: country
Unique Values: 217
country
Afghanistan              27
Albania                  27
Algeria                  27
American Samoa           27
Andorra                  27
                         ..
Virgin Islands (U.S.)    27
West Bank and Gaza       27
Yemen, Rep.              27
Zambia                   27
Zimbabwe                 27
Name: count, Length: 217, dtype: int64

Column: region
Unique Values: 5
region
Asia             1782
Europe           1566
Africa           1296
South America    1134
North America      81
Name: count, dtype: int64

Column: economic_tier
Unique Values: 4
economic_tier
high income     2322
upper middle    1593
lower middle    1269
low              675
Name: count, dtype: int64


In [13]:
# Duplicate Country-Year records
duplicates = df[df.duplicated(subset=['country', 'year'], keep=False)]

if duplicates.empty:
    print("✅ No duplicate Country-Year combinations found.")
else:
    print("❌ Duplicate Country-Year combinations found.")
    display(duplicates)

✅ No duplicate Country-Year combinations found.


In [14]:
# to check every country has 27 years of data from 2000 to 2026
country_years = df.groupby('country')['year'].nunique()

print(country_years.value_counts())

year
27    217
Name: count, dtype: int64


In [15]:
# to check every year has 217 country
year_country = df.groupby('year')['country'].nunique()

print(year_country)

year
2000    217
2001    217
2002    217
2003    217
2004    217
2005    217
2006    217
2007    217
2008    217
2009    217
2010    217
2011    217
2012    217
2013    217
2014    217
2015    217
2016    217
2017    217
2018    217
2019    217
2020    217
2021    217
2022    217
2023    217
2024    217
2025    217
2026    217
Name: country, dtype: int64


In [16]:
# to check if all ranks are unique every year
rank_cols = [
    'happiness_rank',
    'global_hunger_rank',
    'human_development_rank',
    'gdp_per_capita_rank',
    'life_expectancy_rank',
    'corruption_perception_rank',
    'democracy_rank',
    'gini_rank',
    'press_freedom_rank',
    'global_peace_rank',
    'environmental_performance_rank'
]

for col in rank_cols:
    duplicates = (
        df.groupby('year')[col]
          .apply(lambda x: x.duplicated().sum())
    )

    if duplicates.sum() == 0:
        print(f"✅ {col}: All ranks are unique every year.")
    else:
        print(f"❌ {col}: Duplicate ranks found.")
        print(duplicates[duplicates > 0])

✅ happiness_rank: All ranks are unique every year.
❌ global_hunger_rank: Duplicate ranks found.
year
2000     16
2001     19
2002     22
2003     29
2004     35
2005     40
2006     41
2007     51
2008     53
2009     59
2010     62
2011     65
2012     66
2013     70
2014     74
2015     78
2016     81
2017     84
2018     86
2019     88
2020     83
2021     86
2022     97
2023    100
2024    103
2025    103
2026    108
Name: global_hunger_rank, dtype: int64
❌ human_development_rank: Duplicate ranks found.
year
2000    30
2001    29
2002    25
2003    22
2004    20
2005    19
2006    19
2007    16
2008    12
2009     9
2010     7
2011     5
2012     3
2013     3
2014     4
2015     3
2016     4
2017     3
2018     2
2019     1
2020     1
2021     2
2022     2
Name: human_development_rank, dtype: int64
❌ gdp_per_capita_rank: Duplicate ranks found.
year
2000    10
2001     9
2002     5
2003     5
2004     5
2005     5
2006     4
2007     4
2008     3
2009     2
2010     2
2011     1
201

* in above duplicate rank are allowed (it means the column where have duplicate rank found it it worth no need to change over here)

In [18]:
df.to_csv('clean_dataset.csv' , index=False)